In [1]:
import sys
import demes
import moments
import numpy as np
import matplotlib.pyplot as plt

from time import time
from momi3.MOMI import Momi

sys.path.append("..")
sys.path.append("../slurm_performance_tests")

/mnt/turbo/eneswork/jthlab/momi3/src/momi3/utils.py:22: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
from Timing_util import total_variation

In [3]:
def get_esfs_moments(demo, sampled_demes, sample_sizes):
    esfs = moments.Spectrum.from_demes(
        demo, sampled_demes=sampled_demes, sample_sizes=sample_sizes
    )
    return esfs * 4 * demo.demes[0].epochs[0].start_size

def get_esfs_momi3(demo, sampled_demes, sample_sizes):
    momi = Momi(demo, sampled_demes=sampled_demes, sample_sizes=sample_sizes, jitted=True)
    return momi.sfs_spectrum() 

In [5]:
ESFS = {'moments': {}, 'momi3':{}}
Times = {'moments': {}, 'momi3':{}}


for ss in [5, 20]:
    for ndemes in [2, 3, 4, 5]:
        if (ndemes > 3) & (ss == 20):
            pass
        else:
            print(ss, ndemes)
            demo = demes.load(f'timing_tests_yaml/{ndemes}_pop_mig.yaml')
            sampled_demes = demo.metadata['sampled_demes']
            sample_sizes = len(sampled_demes) * [ss]

            t = time()
            mesfs = get_esfs_moments(demo, sampled_demes, sample_sizes)
            t = time() - t
            Times['moments'][ss, ndemes] = t
            ESFS['moments'][ss, ndemes] = mesfs

            t = time()
            mesfs = get_esfs_momi3(demo, sampled_demes, sample_sizes)
            t = time() - t
            Times['momi3'][ss, ndemes] = t
            ESFS['momi3'][ss, ndemes] = mesfs        

5 2
5 3
5 4
5 5
20 2
20 3


In [6]:
import pickle
with open("dadi_acc/times_moments_momi3.pickle", "wb") as f:
  # Pickle the dictionary
  pickle.dump(Times, f)
    
with open("dadi_acc/esfs_moments_momi3.pickle", "wb") as f:
  # Pickle the dictionary
  pickle.dump(ESFS, f)